In [ ]:
import json
from pathlib import Path
import numpy as np
from nd2reader import ND2Reader
from nd2 import ND2File
from operator import sub
from calmutils.imageio.tiff_imagej import save_tiff_imagej

try:
    from dask_image.ndinterp import affine_transform
    print('will use dask-image for image transformation')
except ImportError:
    from scipy.ndimage import affine_transform
    print('will use scipy for image transformation, consider dask-image for higher speed')

def world_coordinate_transform_to_pixel(transform_matrix, pixel_size):
    pixel_scale_mat = np.diag(list(pixel_size) + [1])
    mat = np.linalg.inv(pixel_scale_mat) @ transform_matrix @ pixel_scale_mat
    return mat

# Correct Chromatic Aberrations for images

This notebook will use the channel-to-channel transformations estimated with ```chromatic_aberration_estimation{_elastix}.ipynb``` and apply them to new images.

We need:
1. the saved JSON transform information from the estimation recipes
2. image files to transform

**Input**
1. path to a directory containing nd2 files
2. path to the saved transforms
3. path to write aligned images to
4. **Parameters**: which channel to use as reference, optionally channel name map if the names differ in JSON and the metadata of new images

This recipe will produce:
* aligned images saved as multichannel TIFF files that can be read by ImageJ

In [ ]:
in_path = Path('/Users/david/Desktop/23AM09-03')
out_path = Path('/Users/david/Desktop/23AM09-03/aligned')
transforms_path = Path('/Users/david/Desktop/23AM09-03/23AM09-03_003_channel_registration.json')

# which channel the coordinates should be aligned to
reference_channel = '405 CSU-W1'

### Channel Renaming
# if the channel names in the JSON transform file and the new files differ
# e.g. if the OC names in NIS were different or images were resaved and just have channel 0, 1, ...,
# we have to rename the channels from the JSON file to match the ones in new images
# the channel alias map should have the form: name in JSON -> name in new images
# channel_aliases = {
#     '405 CSU-W1': '405-CSU-W1',
#     '488 CSU-W1': '488-CSU-W1',
#     '561 CSU-W1': '561-CSU-W1',
#     '640 CSU-W1': '640-CSU-W1'
# }

# if you do not want to rename the channels, just use an empty dict
channel_aliases = {}

# spinning disk data may have additional magnification of 1.5x
# leave at 1.0 unless you are sure you used the extra zoom
magnification = 1.0

In [ ]:
with open(transforms_path) as fd:
    transform_info = json.load(fd)

# transforms are saved as list of dicts containing channel pair and (flat) parameters
# build dict channel pair -> transform matrix
transforms = {}
for transform_info_i in transform_info['transforms']:
    
    tr = np.array(transform_info_i['parameters']).reshape(4,4)
    
    # apply channel renaming if necessary
    channels = map(lambda c: channel_aliases[c] if c in channel_aliases else c, transform_info_i['channels'])

    transforms[tuple(channels)] = tr

in_files = sorted(in_path.glob('*.nd2'))
in_files

In [ ]:
if not out_path.exists():
    out_path.mkdir()

for image_path in in_files:

    print(f'aligning {image_path}')

    # read all channels into dict of channel_name -> img
    images = {}
    with ND2File(image_path) as reader:

        for i, channel in enumerate(reader.metadata.channels):
            img = np.array(reader.to_dask()[:,i])
            images[channel.channel.name.strip()] = img
        # invert xyz voxel size to zyx to match img array
        pixel_size = np.array(reader.voxel_size()[::-1])

    images_aligned = {}
    for ch, image in images.items():

        if ch == reference_channel:
            images_aligned[ch] = image
            print(f'keep image of channel {ch} as-is (reference)')
            continue
            
        # NOTE: we want the inverse transform from ch to reference, i.e. the transform reference -> ch
        mat = transforms[(reference_channel, ch)]
        mat = world_coordinate_transform_to_pixel(mat, pixel_size)

        image_transformed = affine_transform(image, mat, order=2)
        images_aligned[ch] = np.array(image_transformed)

        print(f'aligned image of channel {ch}')

    # save
    images_aligned_stacked = np.stack(list(images_aligned.values()))
    out_file = out_path / (image_path.stem + '_aligned.tif')
    save_tiff_imagej(out_file, images_aligned_stacked, axes='czyx', pixel_size=pixel_size, distance_unit='micron')

    print(f'finished aligning {image_path}')